<table align="center">
  <!-- ASSTIA link -->
  <td align="center">
    <a target="_blank" href="https://trilobit-coder.github.io/deeplearning/">
      <img src="../docs/assets/logo.png" width="110px" height="70px" style="padding-bottom:5px;" />
      Visit SSTIA DeepLearning
    </a>
  </td>

  <!-- Google Colab link -->
  <td align="center">
    <a target="_blank" href="https://colab.research.google.com/github/trilobit-coder/deeplearning/blob/main/model/NanoShakespeare.ipynb">
      <img src="https://i.ibb.co/2P3SLwK/colab.png" width="110px" height="70px" style="padding-bottom:5px;" />
      Run in Google Colab
    </a>
  </td>
  
  <!-- GitHub source link -->
  <td align="center">
    <a target="_blank" href="https://github.com/trilobit-coder/deeplearning/blob/main/model/NanoShakespeare.ipynb">
      <img src="https://i.ibb.co/xfJbPmL/github.png" width="70px" height="70px" style="padding-bottom:5px;" />
      View Source on GitHub
    </a>
  </td>
</table>

## Generate Shakespearean Text With RNN

Runing this code in colab. To get faster calculation speed, click "Runtime" at the top bar and choose "Change runtime type"-"T4 GPU"

### 14.1 Dependencies

In [ ]:
! wget https://raw.githubusercontent.com/trilobit-coder/deeplearning/main/docs/assets/ShakespeareData.txt

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time

### 14.2 Hyperparameters

In [ ]:
# --- HYPERPARAMETERS ---

batch_size = 32
block_size = 256
max_iters = 2000
eval_interval = 500
learning_rate = 6e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 100
n_embd = 384
n_layer = 3
dropout = 0.5

# ----------------------------------------------------

torch.manual_seed(1337)

### 14.3 Data Preparation

In [ ]:
# Load data

with open("ShakespeareData.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

# Move data to device

data = torch.tensor(encode(text), dtype=torch.long).to(device)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i : i + block_size] for i in ix])
    y = torch.stack([data_source[i + 1 : i + block_size + 1] for i in ix])
    return x, y


### 14.4 LSTM Model

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        self.lstm = nn.LSTM(
            input_size=n_embd,
            hidden_size=n_embd,
            num_layers=n_layer,
            dropout=dropout,
            batch_first=True,
        )
        self.ln = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None, hidden=None):
        x = self.token_embedding_table(idx)
        x, hidden = self.lstm(x, hidden)
        x = self.ln(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss, hidden

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # Initial context processing
        logits, _, hidden = self(idx)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_input = torch.multinomial(probs, num_samples=1)
        generated = torch.cat((idx, next_input), dim=1)

        # Stateful loop
        for _ in range(max_new_tokens - 1):
            logits, _, hidden = self(next_input, hidden=hidden)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_input = torch.multinomial(probs, num_samples=1)
            generated = torch.cat((generated, next_input), dim=1)
        return generated

model = LSTMLanguageModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

### 14.5 Training Loop

In [ ]:
def estimate_loss(model):
    """Helps us see how the model is doing on both train and val data"""
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss, _ = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print(f"Training on {device}...")
start_time = time.time()

for iter in range(max_iters):
    # --- VALIDATION STEP ---
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        print(
            f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f} | time: {elapsed:.2f}s"
        )

    xb, yb = get_batch("train")
    logits, loss, _ = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

### 14.6 Generation

The text may look like Shakespeare but make no sense. This is because the model needs larger dataset for traing to truly make something awesome due to timing. But you can simply try to improve it but change the hyperparameters.

In [ ]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- GENERATED TEXT ---\n")
print(decode(model.generate(context, max_new_tokens=1500)(0).tolist()))